# 第 5 章 パーセプトロン

点を直線で 2 クラスに分けます。誤分類した点だけを使ってモデルを動かすパーセプトロントリックを確かめます。

対応する記事: [第 5 章 パーセプトロン（Kotlin Notebook の言語版）](../../../docs/article/grokking-machine-learning/kotlin/ch05.md)

実装本体: `apps/grokking-ml-kotlin/src/`

## セットアップ

実装本体をビルドした JAR を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

先に JAR を作っておいてください。

```bash
cd apps/grokking-ml-kotlin
./gradlew jar
```

IntelliJ IDEA の Kotlin Notebook プラグイン、または [Kotlin Jupyter カーネル](https://github.com/Kotlin/kotlin-jupyter) で開きます。

```bash
pip install kotlin-jupyter-kernel
jupyter lab notebooks/
```

In [1]:
@file:DependsOn("../build/libs/grokking-ml-kotlin-0.1.0.jar")

import ch05.*

## データセット

原著と同じ「悲しい／楽しい」文の分類データです。特徴量は単語 `aack` と `beep` の出現回数、ラベルは 1 が「楽しい」です。

In [2]:
val points = listOf(listOf(1.0, 0.0), listOf(0.0, 2.0), listOf(1.0, 1.0), listOf(1.0, 2.0),
                    listOf(1.0, 3.0), listOf(2.0, 2.0), listOf(2.0, 3.0), listOf(3.0, 2.0))
val labels = listOf(0, 0, 0, 0, 1, 1, 1, 1)

points.zip(labels).forEach { (point, label) ->
    println("aack=%.0f beep=%.0f → %s".format(point[0], point[1], if (label == 1) "楽しい" else "悲しい"))
}

aack=1 beep=0 → 悲しい
aack=0 beep=2 → 悲しい
aack=1 beep=1 → 悲しい


aack=1 beep=2 → 悲しい
aack=1 beep=3 → 楽しい
aack=2 beep=2 → 楽しい


aack=2 beep=3 → 楽しい
aack=3 beep=2 → 楽しい


## トリックは誤分類した点だけを動かす

**正しく分類できている点では、モデルがまったく変わりません。** これが第 6 章のロジスティック回帰との決定的な違いです。

In [3]:
val model = Perceptron(listOf(1.0, 2.0), -4.0)
println("元のモデル       $model")
println("正解した点を渡す " + perceptronTrick(model, listOf(1.0, 2.0), label = 1, learningRate = 0.1))
println("誤分類の点を渡す " + perceptronTrick(model, listOf(1.0, 1.0), label = 1, learningRate = 0.1))

元のモデル       Perceptron(weights=[1.0, 2.0], bias=-4.0)


正解した点を渡す Perceptron(weights=[1.0, 2.0], bias=-4.0)


誤分類の点を渡す Perceptron(weights=[1.1, 2.1], bias=-3.9)


## 学習

In [4]:
val (trained, errors) = perceptronAlgorithm(points, labels, learningRate = 0.01, epochs = 1000, seed = 0)

println("重み   " + trained.weights.map { "%.4f".format(it) })
println("バイアス %.4f".format(trained.bias))
println("正解率  %.2f".format(accuracy(trained, points, labels)))

重み   [0.0100, 0.0100]
バイアス -0.0300
正解率  1.00


## パーセプトロン誤差の落とし穴

**初期状態（全パラメータ 0）の誤差は 0 です。** すべての点が境界線上にあるため、誤分類していてもスコアの絶対値が 0 だからです。正解率は 0.5 しかないのに、誤差関数は最良と報告します。

学習の進み具合を見るなら、誤差ではなく **正解率** を見るべきです。

In [5]:
val initial = Perceptron(listOf(0.0, 0.0), 0.0)
println("初期の平均誤差 %.4f".format(meanPerceptronError(initial, points, labels)))
println("初期の正解率   %.2f".format(accuracy(initial, points, labels)))
println()
println("学習中の最大誤差 %.4f".format(errors.max()))
println("最終の平均誤差   %.4f".format(errors.last()))
println("最終の正解率     %.2f".format(accuracy(trained, points, labels)))

初期の平均誤差 0.0000
初期の正解率   0.50

学習中の最大誤差 0.0275


最終の平均誤差   0.0000
最終の正解率     1.00


## 試してみる

分類では **重みの絶対値に意味がありません**。比率と符号だけが境界線を決めます。すべてを 100 倍しても、予測はまったく同じです。

In [6]:
val scaled = Perceptron(trained.weights.map { it * 100 }, trained.bias * 100)
println("元のモデル    " + points.map { trained.predict(it) })
println("100 倍モデル  " + points.map { scaled.predict(it) })
println("正解          " + labels)

元のモデル    [0, 0, 0, 0, 1, 1, 1, 1]
100 倍モデル  [0, 0, 0, 0, 1, 1, 1, 1]


正解          [0, 0, 0, 0, 1, 1, 1, 1]
